# 02 - Cross-validation folds (v2)

Goal: assign every **training** row to one of 5 validation folds, once, and save the assignment so every model uses exactly the same folds.

- 5-fold `StratifiedKFold`, stratified on income x sex, shuffled.
- Repeated 3 times with seeds 42, 43, 44 (fixed in advance): `fold_r0`, `fold_r1`, `fold_r2`.
- `fold_r0` (seed 42) is the single run used for exploration; all three are used for the final comparison.
- Test rows are never used here. `data/adult.csv`, `results/split_indices.csv` and `results/dropped_duplicates.csv` are only read.
- v2: built on the de-duplicated split from `01_split.ipynb`. v1 is archived in `notebooks/archive_v1/` and `results/archive_v1/`.

## 1. Kernel and library versions
`sys.executable` shows which Python runs this notebook. It should be `C:\Users\User\venvs\classification\Scripts\python.exe` (kernel "Python (classification)"). If it is anything else, stop and switch the kernel.

In [7]:
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold

print('Python executable:', sys.executable)
print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scikit-learn', sklearn.__version__)

Python executable: c:\Users\User\venvs\classification\Scripts\python.exe
pandas       3.0.6
numpy        2.4.6
scikit-learn 1.9.1


## 2. Load the raw data, the split and the dropped duplicates (read-only)
- Same root detection and same `read_csv` arguments as `01_split.ipynb`: `header=None`, the 15 names, `keep_default_na=False`.
- The v2 split lists only the **kept** rows. Its `row_id`s must be exactly all positions 0 ... 48,841 **minus** the `row_id`s in `dropped_duplicates.csv`: 48,790 entries, each once.
- Split labels must be only `train` and `test`. The train and test sizes are printed, not compared with fixed numbers; later cells check against these sizes.

In [8]:
ROOT = Path.cwd()
if not (ROOT / 'data' / 'adult.csv').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'adult.csv'
RESULTS_PATH = ROOT / 'results'
SPLIT_PATH = RESULTS_PATH / 'split_indices.csv'
DROPPED_PATH = RESULTS_PATH / 'dropped_duplicates.csv'

for path in [DATA_PATH, SPLIT_PATH, DROPPED_PATH]:
    assert path.exists(), f'Not found: {path}'
print('Project root:', ROOT)

COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income',
]

df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS, keep_default_na=False)
print('Raw data shape:', df.shape)
assert df.shape == (48842, 15), f'Unexpected shape {df.shape}'

split_df = pd.read_csv(SPLIT_PATH)
assert list(split_df.columns) == ['row_id', 'split'], f'Unexpected columns {list(split_df.columns)}'
assert split_df['row_id'].is_unique, 'row_id has duplicates'

dropped_df = pd.read_csv(DROPPED_PATH)
assert list(dropped_df.columns) == ['row_id', 'kept_row_id'], f'Unexpected columns {list(dropped_df.columns)}'
assert dropped_df['row_id'].is_unique, 'dropped row_id has duplicates'
print('Dropped duplicates listed:', len(dropped_df))

expected_ids = set(range(len(df))) - set(dropped_df['row_id'])
assert set(split_df['row_id']) == expected_ids, 'Split row_ids are not exactly all positions minus the dropped duplicates'
assert len(split_df) == 48790, f'Split has {len(split_df)} rows, expected 48790'

split_labels = set(split_df['split'].unique())
assert split_labels == {'train', 'test'}, f'Unexpected split labels {split_labels}'

split_counts = split_df['split'].value_counts()
N_TRAIN = int(split_counts['train'])
N_TEST = int(split_counts['test'])
print(f'Train rows: {N_TRAIN:,}')
print(f'Test rows:  {N_TEST:,}')

Project root: c:\Users\User\OneDrive\Desktop\Classification_Model
Raw data shape: (48842, 15)
Dropped duplicates listed: 52
Train rows: 39,032
Test rows:  9,758


## 3. Keep only the training rows
- Train rows are taken **sorted by `row_id`**. StratifiedKFold's result depends on row order, so a fixed, documented order makes the folds reproducible.
- Only the list of test `row_id`s is kept (no test data), for the leakage check in section 5. The full raw table is deleted.
- The stratification label is the same four groups as in Step 2, e.g. `>50K_Female`.

In [9]:
train_ids = np.sort(split_df.loc[split_df['split'] == 'train', 'row_id'].to_numpy())
test_ids = np.sort(split_df.loc[split_df['split'] == 'test', 'row_id'].to_numpy())

train_df = df.iloc[train_ids].reset_index(drop=True)
del df  # from here on, only training rows exist in memory
assert len(train_df) == N_TRAIN

income_values = set(train_df['income'].unique())
sex_values = set(train_df['sex'].unique())
assert income_values == {'<=50K', '>50K'}, f'Unexpected income values: {income_values}'
assert sex_values == {'Male', 'Female'}, f'Unexpected sex values: {sex_values}'

strat = train_df['income'] + '_' + train_df['sex']
print(strat.value_counts())

<=50K_Male      18161
<=50K_Female    11526
>50K_Male        7930
>50K_Female      1415
Name: count, dtype: int64


## 4. Assign folds: 3 repeats x 5 folds
For each seed, `StratifiedKFold` yields 5 (train, validation) pairs. We record, for every row, the number (0-4) of the fold in which it is a **validation** row. We also count how often each row is a validation row, to check it is exactly once.
The features are not needed to make folds, so `X` is a dummy array; only the stratification label matters.

In [10]:
SEEDS = [42, 43, 44]
N_SPLITS = 5
FOLD_COLS = [f'fold_r{r}' for r in range(len(SEEDS))]

X_dummy = np.zeros((len(train_df), 1))
folds = pd.DataFrame({'row_id': train_ids})
times_in_val = {}

for col, seed in zip(FOLD_COLS, SEEDS):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    fold = np.full(len(train_df), -1, dtype=int)
    count = np.zeros(len(train_df), dtype=int)
    for k, (_, val_idx) in enumerate(skf.split(X_dummy, strat)):
        fold[val_idx] = k
        count[val_idx] += 1
    folds[col] = fold
    times_in_val[col] = count

print(folds.head())

   row_id  fold_r0  fold_r1  fold_r2
0       0        2        1        4
1       1        2        0        4
2       2        0        2        2
3       3        3        2        2
4       4        0        2        4


## 5. Checks per repeat
- Every train row is a validation row exactly once, and all 5 fold numbers are used.
- No test `row_id` is among the CV rows (leakage check).
- Printed for reading: fold sizes, the >50K share per fold, and the four-group shares per fold. If stratification worked, each fold's shares should be very close to the overall train shares printed first.

In [11]:
assert len(folds) == N_TRAIN, f'CV rows {len(folds)}, expected {N_TRAIN}'
n_test_in_cv = int(np.isin(folds['row_id'], test_ids).sum())
assert n_test_in_cv == 0, f'{n_test_in_cv} test row_ids are in the CV folds'
print('Test row_ids in CV folds:', n_test_in_cv)

is_high = (train_df['income'] == '>50K')
print(f'Overall train >50K share: {is_high.mean():.4%}')
print('Overall train four-group shares (%):')
print((strat.value_counts(normalize=True) * 100).round(3).to_string())

for col, seed in zip(FOLD_COLS, SEEDS):
    print()
    print(f'=== {col} (random_state={seed}) ===')
    assert (folds[col] >= 0).all(), f'{col}: some rows got no fold'
    assert (times_in_val[col] == 1).all(), f'{col}: some rows are validation rows more or less than once'
    assert set(folds[col].unique()) == set(range(N_SPLITS)), f'{col}: not all fold numbers used'

    print('Fold sizes:')
    print(folds[col].value_counts().sort_index().to_string())

    print('>50K share per fold (%):')
    print((is_high.groupby(folds[col]).mean() * 100).round(3).to_string())

    print('Four-group shares per fold (%):')
    print((pd.crosstab(folds[col], strat, normalize='index') * 100).round(3))

print()
print('All checks passed.')

Test row_ids in CV folds: 0
Overall train >50K share: 23.9419%
Overall train four-group shares (%):
<=50K_Male      46.528
<=50K_Female    29.530
>50K_Male       20.317
>50K_Female      3.625

=== fold_r0 (random_state=42) ===
Fold sizes:
fold_r0
0    7807
1    7807
2    7806
3    7806
4    7806
>50K share per fold (%):
fold_r0
0    23.940
1    23.940
2    23.943
3    23.943
4    23.943
Four-group shares per fold (%):
col_0    <=50K_Female  <=50K_Male  >50K_Female  >50K_Male
fold_r0                                                  
0              29.525      46.535        3.625     20.315
1              29.538      46.522        3.625     20.315
2              29.529      46.528        3.625     20.318
3              29.529      46.528        3.625     20.318
4              29.529      46.528        3.625     20.318

=== fold_r1 (random_state=43) ===
Fold sizes:
fold_r1
0    7807
1    7807
2    7806
3    7806
4    7806
>50K share per fold (%):
fold_r1
0    23.940
1    23.940
2    23.94

## 6. The three repeats really differ
Repeating CV only helps if each repeat shuffles the rows differently. For each pair of repeats we compute the share of rows that got the **same fold number** in both.
- Two independent random 5-fold assignments agree on about 1/5 = 20% of rows by chance.
- A share near 100% would mean the seeds did not take effect (the repeats are copies). We assert the share is below 0.5.

In [12]:
for a, b in combinations(FOLD_COLS, 2):
    agree = (folds[a] == folds[b]).mean()
    print(f'{a} vs {b}: same fold number for {agree:.2%} of rows')
    assert agree < 0.5, f'{a} and {b} agree on {agree:.2%} of rows; the repeats are not independent'

print('Repeats differ as expected.')

fold_r0 vs fold_r1: same fold number for 20.48% of rows
fold_r0 vs fold_r2: same fold number for 20.13% of rows
fold_r1 vs fold_r2: same fold number for 20.01% of rows
Repeats differ as expected.


## 7. Save the fold assignment
Columns `row_id, fold_r0, fold_r1, fold_r2`, train rows only, sorted by `row_id`. `row_id` is the same 0-based position in `data/adult.csv` as in `split_indices.csv`.
If `results/cv_folds.csv` already exists, it is **not overwritten**: we load it and assert it is identical to what we just computed. That is also a reproducibility check (e.g. after a library update).

In [13]:
out = folds[['row_id'] + FOLD_COLS].sort_values('row_id').reset_index(drop=True).astype('int64')
out_path = RESULTS_PATH / 'cv_folds.csv'

if out_path.exists():
    existing = pd.read_csv(out_path)
    pd.testing.assert_frame_equal(existing, out, check_dtype=False)
    print('File already exists and is identical; not overwritten.')
else:
    RESULTS_PATH.mkdir(exist_ok=True)
    out.to_csv(out_path, index=False)
    print('Saved:', out_path)

print('Path:', out_path)
print('Rows:', len(out))

Saved: c:\Users\User\OneDrive\Desktop\Classification_Model\results\cv_folds.csv
Path: c:\Users\User\OneDrive\Desktop\Classification_Model\results\cv_folds.csv
Rows: 39032
